In [ ]:
# Author: Niko Bleidistel
# last change: 2026-08-04

# Package Import

In [ ]:
from pathlib import Path 
from os import makedirs
import sys
import importlib
import re

import pandas as pd
import numpy as np

import mph
import matplotlib as mpl
import matplotlib.pyplot as plt

import time
import logging
import csv

In [ ]:
PYTHON_HELPER_FOLDER = Path(r"py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import comsol_data_export as cde
import comsol_data_plotting as cdp
import plot_functions as pfs
import time_logging as tl

# reload custom packages (for each execution) to reflect recent changes
_ = importlib.reload(cde)
_ = importlib.reload(cdp)
_ = importlib.reload(pfs)
_ = importlib.reload(tl)

# PATHS

In [ ]:
# INPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\06_mfco_assymmetry")
# OUTPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Script Outputs\000_New_Output")

MAIN_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\10_bachelor_thesis_models_use_terminals")
INPUT_FOLDER = MAIN_FOLDER / "Solved model versions"
OUTPUT_FOLDER = MAIN_FOLDER / "Test Output"

makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

# INITIALIZE

In [ ]:
# initialize time logging
_ = tl.initialize_time_log(OUTPUT_FOLDER / 'time_log.csv')

# initialize COMSOL client (server)
client = mph.start()

# SIMULATE

## just solving model

In [ ]:
# just solve the models from a input folder and save the solved models to an output folder in the same directory 
if False:
    input_folder = MAIN_FOLDER
    output_folder = INPUT_FOLDER
    makedirs(output_folder, exist_ok=True)  # create output folder if it doesn't exist

    mph_files = list(input_folder.glob("*.mph"))

    print("List of model files to be processed:")
    for modelfile in mph_files:
            print(modelfile.stem)

    print("\nCurrently processing...\n")
    for modelfile in mph_files:
        print(modelfile.stem)
        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = output_folder,
            
                    # simulation settings
                    client = client,
                    export_params = [],
                    export_descriptions = [],
            
                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    export_all_solution_data = False,
                    save_small_model_version = False,
                    new_log_file = True,
                )
        except Exception as e:
            with open(output_folder / 'error_log.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")
        print(f"Finished.\n")

## Constants

In [ ]:
EXPORT_DICT = {
    "mf.normB": "Magnetic flux density, norm [T]",
    "mf.Bx": "Magnetic flux density, x-component [T]", 
    "mf.By": "Magnetic flux density, y-component [T]", 
    "mf.Bz": "Magnetic flux density, z-component [T]",
    "T": "Temperature [K]",
}
EXPORT_PARAMS = list(EXPORT_DICT.keys())
EXPORT_DESCRIPTION = list(EXPORT_DICT.values())

## Simulate

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    mph_files = list(input_folder.glob("*.mph"))

    print("List of model files to be processed:")
    for modelfile in mph_files:
            print(modelfile.stem)

    print("\nCurrently processing...\n")
    for modelfile in mph_files:
        print(modelfile.stem)

        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,
                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*substrate_length", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*substrate_length", 0.0),
                    Homogeneity_distances = "range(0,(-1*epilayer_height-0)/9,-1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*substrate_length", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*substrate_length", 0.0, 0.0),
                    Longitudinal_distances = "range(0,(-1*epilayer_height-0)/9,-1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= True,
                    show_model_info = False,
                    solve_model = False, # already solved models are used here
                    save_solved_model = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    save_small_model_version = False,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'error_log.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")
        print(f"Finished.\n")

List of model files to be processed:
01_03_b-Rectangular spiral combined with grid_solved

Currently processing...

01_03_b-Rectangular spiral combined with grid_solved
Finished.



## TODO: SWEEP FUNCTION

# PLOT

## Helper Functions

In [ ]:
def import_solution_data(
        # path settings
        filename: str, # f"{modelname}_exported_data.txt"
        input_folder: Path,
        ):

    txt_path = input_folder / filename
    tl.log_message(f"Started interpolating workflow for exported solution data in {txt_path.name}")

    
    # import data from txt file
    header_data, df = cdp.read_comsol_export(str(txt_path))
    constants_df = cde.find_constant_columns(df)

    # get original modelname
    modelstem = str(header_data.get('Model')).replace(".mph", "")
    tl.log_message(f"Imported data shows original modelname {modelstem}")

    return header_data, df, constants_df, modelstem

## Constants

In [ ]:
X_AXIS_PARAMS = ["x", "y", "z"]
Y_AXIS_PARAMS = ["mf.normB (T)", "mf.Bx (T)", "mf.By (T)", "mf.Bz (T)", "T (K)",]

In [ ]:
TRANSLATE_PLOTLABELS = {
    "mf.normB (T)": "Magnetic flux density, norm [T]",
    "mf.Bx (T)": "Magnetic flux density, x-component [T]", 
    "mf.By (T)": "Magnetic flux density, y-component [T]", 
    "mf.Bz (T)": "Magnetic flux density, z-component [T]",
    "x": "longitudinal (x-axis) [m]",
    "y": "width (y-axis) [m]",
    "z": "depth (z-axis) [m]",
    "T (K)": "Temperature [K]",
}

## OLD: Interpolate

In [ ]:
if False:
    input_folder = OUTPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    export_data_files = list(input_folder.rglob("*_exported_data.txt"))

    print("List of exported data files to be processed:")
    for txtfile in export_data_files:
        print(txtfile.stem)

    print("\nCurrently processing...\n")
    for txtfile in export_data_files:
        print(txtfile.stem)
        try:
            header_data, df, constants_df, modelstem = import_solution_data(
                filename = txtfile.name,
                input_folder = txtfile.parent,
            )
            constants_df.to_csv(output_folder / f"{modelstem}_constant_columns.csv", index=False)

            tl.log_message(f"Searching columns height, width and length in the imported data")
            try:
                height = df['root.epilayer_height (m)'].iloc[0]
                width = 0.5 * df['root.conductor_all_width (m)'].iloc[0]
                length = 0.5 * df['root.conductor_all_length (m)'].iloc[0]
            except KeyError as e:
                tl.log_message(f"ERROR appeared while searching for height, width and length in the imported data")
                height = 1e-6
                width = 0.5 * 500e-6
                length = 0.5* 500e-6
                tl.log_message(f"Using substitute values: height: {height}, width: {width}, length: {length}")

        # DEPTH INTERPOLATION
            print(f"Depth, ")
            interpolate_and_save(
                # path settings
                modelname = modelstem,
                output_folder = output_folder,
                interpolation_type = "depth",

                # grid settings
                grid_dict = {
                    'x': [0.0],
                    'y': [0.0],
                    },

                limit_dict = {
                    'z': [0, - height],
                    },

                # data settings
                df = df,
                x_axis_params = X_AXIS_PARAMS,
                y_axis_params = Y_AXIS_PARAMS,
                )

        # HOMOGENEITY INTERPOLATION
            print(f"Homogeneity, ")
            interpolate_and_save(
                # path settings
                modelname = modelstem,
                output_folder = output_folder,
                interpolation_type = "homogeneity",

                # grid settings
                grid_dict = {
                    'x': [0.0],
                    'z': np.linspace(0, -height, 11),
                    },

                limit_dict = {
                    'y': [1.2*width, -1.2*width],
                    },

                # data settings
                df = df,
                x_axis_params = X_AXIS_PARAMS,
                y_axis_params = Y_AXIS_PARAMS,
                )

        # LONGITUDINAL INTERPOLATION
            print(f"Longitudinal, ")
            interpolate_and_save(
                # path settings
                modelname = modelstem,
                output_folder = output_folder,
                interpolation_type = "longitudinal",

                # grid settings
                grid_dict = {
                    'y': [0.0],
                    'z': np.linspace(0, -height, 11),
                    },

                limit_dict = {
                    'x': [1.2*width, -1.2*width],
                    },

                # data settings
                df = df,
                x_axis_params = X_AXIS_PARAMS,
                y_axis_params = Y_AXIS_PARAMS,
                )

        # ERROR LOGGING
        except Exception as e:
            with open(output_folder / 'error_log.txt', 'a') as f:
                f.write(f"Error occurred while processing {txtfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {txtfile.name}")
        print(f"Finished.\n")

In [ ]:
if False:
    input_folder = OUTPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    interpolated_data_files = list(input_folder.rglob("*_data.csv"))

    print("List of interpolated data files to be processed:")
    for csvfile in interpolated_data_files:
        print(csvfile.stem)

    print("\nCurrently processing...\n")
    for csvfile in interpolated_data_files:
        print(csvfile.stem)
        try:
            #TODO: Plotting function for interpolated data
            # use: output_folder / f"{modelstem}_constant_columns.csv"
            pass
        # ERROR LOGGING
        except Exception as e:
            with open(output_folder / 'error_log.txt', 'a') as f:
                f.write(f"Error occurred while processing {csvfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {csvfile.name}")
        print(f"Finished.\n")

# END

In [ ]:
tl.log_message("Reached the end of the script.")